# Lesson 25 Lab — Benchmarking Sparsity: Proving a Real Speedup

**Puzzle:** Which benchmark prevents a lower mean from hiding an unchanged p99 or memory peak?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Sparse acceleration is a runtime claim. The protocol must freeze shapes, batches, dtype, warm-up, synchronization, sample window, power state, and backend. It should report a distribution and throughput, plus memory and operator evidence, rather than a single average.


## 0. Predict before running

1. Predict which candidate changes dense operator dimensions.
2. Explain why 20 samples are weak evidence for p99.
3. Choose a warm-up and sampling protocol before reading results.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

Dense, same-shape masked, and physically narrowed linear blocks are timed with retained per-iteration CUDA-event samples at batch 1 and batch 64. Median, p95, p99, throughput, and peak memory are computed.

- Benchmark identity includes workload, backend, and timing semantics.
- Mean, median, and tail latency can rank candidates differently.
- Masked dense and physically narrow controls distinguish zeros from less work.


## 2. Derive the mechanism

GPU work is asynchronous, so host timing without synchronization measures enqueue cost. Warm-up absorbs initialization and algorithm selection. Percentiles require sorted repeated samples; `p99` is unstable with too few points. Throughput is workload completed per unit time and should be measured at the serving batch, not derived from peak FLOPs. Peak memory must be reset around each candidate.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 25
LESSON_TITLE = 'Benchmarking Sparsity: Proving a Real Speedup'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260833
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | dense full-width and same-shape 75%-masked dense execution |
| Candidate | physically quarter-width dense execution |
| Held constant | GPU, clocks as observed, shapes, weights, dtype, batches, warm-up, samples, and synchronization |
| Measurements | p50/p95/p99 latency, throughput, peak memory, shape, and speedup |
| Evidence | `pytorch-gpu` |

**Experiment:** Benchmark dense, masked, and narrowed candidates across latency and throughput batches with retained samples.


## 5. Read the experiment code

The timing helper allocates tensors before measurement, warms each candidate, and records individual CUDA-event durations. Summary functions preserve raw samples in the JSON artifact. A separate large batch prevents single-request latency from masquerading as service throughput.

Do not execute until the code implements the frozen table above.


In [2]:
dtype=torch.bfloat16; in_f,out_f=2048,2048
w=torch.randn(out_f,in_f,device=DEVICE,dtype=dtype); masked=w*magnitude_mask(w,0.75); narrow=w[:512]
def bench(weight,batch,repeats=80):
    x=torch.randn(batch,in_f,device=DEVICE,dtype=dtype); torch.cuda.reset_peak_memory_stats(); t=timing_summary(cuda_times(lambda:F.linear(x,weight),warmup=12,repeats=repeats)); peak=torch.cuda.max_memory_allocated()/2**20; return t,peak
d1,dp=bench(w,1); m1,mp=bench(masked,1); n1,np=bench(narrow,1); d64,_=bench(w,64,50); n64,_=bench(narrow,64,50)
metrics={"dense_p50_ms":d1["median_ms"],"dense_p95_ms":d1["p95_ms"],"dense_p99_ms":d1["p99_ms"],"masked_p50_ms":m1["median_ms"],"masked_p95_ms":m1["p95_ms"],"masked_p99_ms":m1["p99_ms"],"narrow_p50_ms":n1["median_ms"],"narrow_p95_ms":n1["p95_ms"],"narrow_p99_ms":n1["p99_ms"],"batch64_dense_ms":d64["median_ms"],"batch64_narrow_ms":n64["median_ms"],"batch64_speedup":d64["median_ms"]/n64["median_ms"],"peak_memory_mib":{"dense":dp,"masked":mp,"narrow":np},"samples":80,"raw_samples":{"dense":d1["samples_ms"],"masked":m1["samples_ms"],"narrow":n1["samples_ms"]}}
analysis=(f"At batch 1, dense p50/p99 were {d1['median_ms']:.6f}/{d1['p99_ms']:.6f} ms, the same-shape masked candidate was "
          f"{m1['median_ms']:.6f}/{m1['p99_ms']:.6f} ms, and the physical narrow candidate was {n1['median_ms']:.6f}/"
          f"{n1['p99_ms']:.6f} ms. At batch 64 the narrow/full median ratio was {metrics['batch64_speedup']:.3f}x.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Dense p50 | 0.017056 ms |
| Dense p99 | 0.022265 ms |
| Masked p50 | 0.017136 ms |
| Narrow p50 | 0.013856 ms |
| Narrow p99 | 0.015994 ms |
| Batch-64 speedup | 0.982x |
| Samples | 80 |


## 7. Interpret rather than merely print

At batch 1, dense p50/p99 were 0.017056/0.022265 ms, the same-shape masked candidate was 0.017136/0.018128 ms, and the physical narrow candidate was 0.013856/0.015994 ms. At batch 64 the narrow/full median ratio was 0.982x.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 25,
    "title": 'Benchmarking Sparsity: Proving a Real Speedup',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'A sparsity speedup is a distribution measured on the intended execution path, not a zero count or a best-case sample.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 25,
  "title": "Benchmarking Sparsity: Proving a Real Speedup",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260833
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "dense_p50_ms": 0.017055999487638474,
    "dense_p95_ms": 0.01863200021907687,
    "dense_p99_ms": 0.02226496037095779,
    "masked_p50_ms": 0.017136000096797943,
    "masked_p95_ms": 0.017635200172662735,
    "masked_p99_ms": 0.01812799971550702,
    "narrow_p50_ms": 0.013856000266969204,
    "narrow_p95_ms": 0.01437280043028295,
    "narrow_p99_ms": 0.015993600133806452,
    "batch64_dense_ms": 0.016847999766469002,
    "batch64_narrow_ms": 0.017152000218629837,
    "batch64_speedup": 0.9822760932669158,
    "peak_memory_mib": {
      "dense": 48.0078125,
      "masked": 48.0078125,
      "narrow": 48.0048828125
    },
    "samples": 80,
    "raw_samples": {
      

## 9. Make the bounded decision

> A sparsity speedup is a distribution measured on the intended execution path, not a zero count or a best-case sample.

**Acceptance/rollback:** Accept a speedup only when the matched baseline, tail gate, throughput gate, memory gate, and operator/shape evidence all meet the frozen protocol.

**Failure analysis:** Shared GPU load, dynamic clocks, allocator history, and insufficient samples can move tails. Microbenchmarks omit data movement and service queues. A narrower toy layer cannot prove an end-to-end model gain.


## 10. Extend the evidence

Repeat in an isolated process, capture Nsight or profiler operator names, add confidence intervals, and run a representative end-to-end service load with request arrivals.

The full evidence boundary and references are in [`README.md`](README.md).
